### Task P2.2
### **Particle Filter**

In [1]:
import numpy as np
import plotly.graph_objs as go

# Constants
G = 9.81  # Gravity (m/s^2)
DT = 0.01  # Time step
N_PART = 1000  # Number of particles
PROC_NOISE = 0.1  # Process noise std dev
OBS_NOISE_X = 0.1  # Observation noise std dev for x
OBS_NOISE_Y = 0.1  # Observation noise std dev for y
INIT_H1 = 20.0  # Initial height of ball 1 (m)
INIT_V1 = 40.0  # Initial speed of ball 1 (m/s)
ANGLE1 = 60.0  # Launch angle of ball 1 (degrees)
INIT_H2 = 20.0  # Initial height of ball 2 (m)
INIT_V2 = 40.0  # Initial speed of ball 2 (m/s)
ANGLE2 = 45.0  # Launch angle of ball 2 (degrees)
SIM_T = 10.0  # Simulation duration (seconds)
NOISE_STD = 0.1  # Observation noise std dev
DROPOUT = 0.05  # Observation dropout probability
VAR_INTERVAL = True  # Variable time intervals

# Simulate Ball Trajectory
def sim_traj(h, v, angle, dt=0.01, t_max=3.0):
    rad = np.radians(angle)
    v_x = v * np.cos(rad)
    v_y = v * np.sin(rad)
    t = np.arange(0, t_max, dt)
    x = v_x * t
    y = h + v_y * t - 0.5 * G * t**2

    ground_hit = np.where(y < 0)[0]
    if len(ground_hit) > 0:
        t = t[:ground_hit[0] + 1]
        x = x[:ground_hit[0] + 1]
        y = y[:ground_hit[0] + 1]

    vx = np.full_like(t, v_x)
    vy = v_y - G * t

    return t, x, y, vx, vy

# Generate Noisy Observations
def noisy_obs(x, y, noise_std=0.1, dropout_rate=0.1, var_time=False, dt=0.01):
    x_obs = x + np.random.normal(0, noise_std, len(x))
    y_obs = y + np.random.normal(0, noise_std, len(y))

    if var_time:
        intervals = np.diff(np.insert(np.cumsum(np.random.exponential(dt, len(x))), 0, 0))
    else:
        intervals = np.full(len(x), dt)

    mask = np.random.rand(len(x)) > dropout_rate
    x_obs = x_obs[mask]
    y_obs = y_obs[mask]
    intervals = intervals[mask]

    return x_obs, y_obs, np.cumsum(intervals)

# Particle Filter
class PartFilter:
    def __init__(self, n_particles, dt, proc_noise, obs_noise_x, obs_noise_y):
        self.n_particles = n_particles
        self.dt = dt
        self.proc_noise = proc_noise
        self.obs_noise_x = obs_noise_x
        self.obs_noise_y = obs_noise_y
        self.particles = np.zeros((n_particles, 4))  # [x, y, vx, vy]
        self.weights = np.ones(n_particles) / n_particles

    def init_particles(self, init_pos, init_speed, angle):
        rad = np.radians(angle)
        self.particles[:, 0] = init_pos[0] + np.random.normal(0, self.proc_noise, self.n_particles)
        self.particles[:, 1] = init_pos[1] + np.random.normal(0, self.proc_noise, self.n_particles)
        self.particles[:, 2] = init_speed * np.cos(rad) + np.random.normal(0, self.proc_noise, self.n_particles)
        self.particles[:, 3] = init_speed * np.sin(rad) + np.random.normal(0, self.proc_noise, self.n_particles)

    def predict(self):
        self.particles[:, 0] += self.particles[:, 2] * self.dt
        self.particles[:, 1] += self.particles[:, 3] * self.dt - 0.5 * G * self.dt**2
        self.particles[:, 2] += np.random.normal(0, self.proc_noise, self.n_particles)
        self.particles[:, 3] += -G * self.dt + np.random.normal(0, self.proc_noise, self.n_particles)

    def update(self, z):
        d = np.linalg.norm(self.particles[:, :2] - z, axis=1)
        self.weights = (1 / np.sqrt(2 * np.pi * self.obs_noise_x**2)) * np.exp(-0.5 * (d**2) / self.obs_noise_x**2)
        self.weights += 1.e-300
        self.weights /= np.sum(self.weights)

    def resample(self):
        idx = np.random.choice(self.n_particles, self.n_particles, p=self.weights)
        self.particles = self.particles[idx]
        self.weights.fill(1.0 / self.n_particles)

    def estimate(self):
        pos = np.average(self.particles[:, :2], weights=self.weights, axis=0)
        vel = np.average(self.particles[:, 2:], weights=self.weights, axis=0)
        return pos[0], pos[1], vel[0], vel[1]

# Generate synthetic data
t1, x1, y1, vx1, vy1 = sim_traj(INIT_H1, INIT_V1, ANGLE1, DT, SIM_T)
noisy_x1, noisy_y1, times1 = noisy_obs(x1, y1, NOISE_STD, DROPOUT, VAR_INTERVAL, DT)

t2, x2, y2, vx2, vy2 = sim_traj(INIT_H2, INIT_V2, ANGLE2, DT, SIM_T)
noisy_x2, noisy_y2, times2 = noisy_obs(x2, y2, NOISE_STD, DROPOUT, VAR_INTERVAL, DT)

# Initialize particle filters for both balls
pf1 = PartFilter(N_PART, DT, PROC_NOISE, OBS_NOISE_X, OBS_NOISE_Y)
pf2 = PartFilter(N_PART, DT, PROC_NOISE, OBS_NOISE_X, OBS_NOISE_Y)

pf1.init_particles([0, INIT_H1], INIT_V1, ANGLE1)
pf2.init_particles([0, INIT_H2], INIT_V2, ANGLE2)

# Apply Particle Filter
est_x1, est_y1, est_vx1, est_vy1 = [], [], [], []
est_x2, est_y2, est_vx2, est_vy2 = [], [], [], []

for i in range(len(times1)):
    pf1.predict()
    pf1.update(np.array([noisy_x1[i], noisy_y1[i]]))
    pf1.resample()
    est1 = pf1.estimate()
    est_x1.append(est1[0])
    est_y1.append(est1[1])
    est_vx1.append(est1[2])
    est_vy1.append(est1[3])

for i in range(len(times2)):
    pf2.predict()
    pf2.update(np.array([noisy_x2[i], noisy_y2[i]]))
    pf2.resample()
    est2 = pf2.estimate()
    est_x2.append(est2[0])
    est_y2.append(est2[1])
    est_vx2.append(est2[2])
    est_vy2.append(est2[3])

# Convert lists to numpy arrays
est_x1 = np.array(est_x1)
est_y1 = np.array(est_y1)
est_vx1 = np.array(est_vx1)
est_vy1 = np.array(est_vy1)
est_x2 = np.array(est_x2)
est_y2 = np.array(est_y2)
est_vx2 = np.array(est_vx2)
est_vy2 = np.array(est_vy2)

# Calculate RMSE for positions
rmse_pos1 = np.sqrt(np.mean((est_x1 - noisy_x1)**2 + (est_y1 - noisy_y1)**2))
rmse_pos2 = np.sqrt(np.mean((est_x2 - noisy_x2)**2 + (est_y2 - noisy_y2)**2))
print(f"RMSE Position Ball 1: {rmse_pos1}")
print(f"RMSE Position Ball 2: {rmse_pos2}")

# Calculate RMSE for velocities
true_vel1 = np.column_stack((vx1[:len(est_vx1)], vy1[:len(est_vy1)]))
rmse_vel1 = np.sqrt(np.mean((np.column_stack((est_vx1, est_vy1)) - true_vel1)**2))
true_vel2 = np.column_stack((vx2[:len(est_vx2)], vy2[:len(est_vy2)]))
rmse_vel2 = np.sqrt(np.mean((np.column_stack((est_vx2, est_vy2)) - true_vel2)**2))
print(f"RMSE Velocity Ball 1: {rmse_vel1}")
print(f"RMSE Velocity Ball 2: {rmse_vel2}")

# Plot results
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=x1, y=y1, mode='lines', name='True Path Ball 1', line=dict(color='blue', width=2)))
fig1.add_trace(go.Scatter(x=noisy_x1, y=noisy_y1, mode='markers', name='Noisy Obs Ball 1', marker=dict(color='orange', size=5, opacity=0.6)))
fig1.add_trace(go.Scatter(x=est_x1, y=est_y1, mode='lines+markers', name='PF Est Ball 1', line=dict(color='green', width=2, dash='dot'), marker=dict(color='green', size=5, opacity=0.6)))

fig1.add_trace(go.Scatter(x=x2, y=y2, mode='lines', name='True Path Ball 2', line=dict(color='cyan', width=2)))
fig1.add_trace(go.Scatter(x=noisy_x2, y=noisy_y2, mode='markers', name='Noisy Obs Ball 2', marker=dict(color='pink', size=5, opacity=0.6)))
fig1.add_trace(go.Scatter(x=est_x2, y=est_y2, mode='lines+markers', name='PF Est Ball 2', line=dict(color='purple', width=2, dash='dot'), marker=dict(color='purple', size=5, opacity=0.6)))

fig1.update_layout(title='Particle Filter: Position Estimates',
                   xaxis_title='Horizontal Position (m)',
                   yaxis_title='Vertical Position (m)',
                   xaxis=dict(tickfont=dict(size=14)),
                   yaxis=dict(tickfont=dict(size=14)),
                   legend=dict(font=dict(size=14)))
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t1[:len(est_vx1)], y=vx1, mode='lines', name='True vx Ball 1', line=dict(color='blue', width=2)))
fig2.add_trace(go.Scatter(x=t1[:len(est_vy1)], y=vy1, mode='lines', name='True vy Ball 1', line=dict(color='orange', width=2)))
fig2.add_trace(go.Scatter(x=times1, y=est_vx1, mode='lines+markers', name='Est vx Ball 1', line=dict(color='green', width=2, dash='dot'), marker=dict(color='green', size=5, opacity=0.6)))
fig2.add_trace(go.Scatter(x=times1, y=est_vy1, mode='lines+markers', name='Est vy Ball 1', line=dict(color='red', width=2, dash='dot'), marker=dict(color='red', size=5, opacity=0.6)))

fig2.add_trace(go.Scatter(x=t2[:len(est_vx2)], y=vx2, mode='lines', name='True vx Ball 2', line=dict(color='cyan', width=2)))
fig2.add_trace(go.Scatter(x=t2[:len(est_vy2)], y=vy2, mode='lines', name='True vy Ball 2', line=dict(color='pink', width=2)))
fig2.add_trace(go.Scatter(x=times2, y=est_vx2, mode='lines+markers', name='Est vx Ball 2', line=dict(color='purple', width=2, dash='dot'), marker=dict(color='purple', size=5, opacity=0.6)))
fig2.add_trace(go.Scatter(x=times2, y=est_vy2, mode='lines+markers', name='Est vy Ball 2', line=dict(color='brown', width=2, dash='dot'), marker=dict(color='brown', size=5, opacity=0.6)))

fig2.update_layout(title='Particle Filter: Velocity Estimates vs. Time',
                   xaxis_title='Time (s)',
                   yaxis_title='Velocity (m/s)',
                   xaxis=dict(tickfont=dict(size=14)),
                   yaxis=dict(tickfont=dict(size=14)),
                   legend=dict(font=dict(size=14)))
fig2.show()


RMSE Position Ball 1: 0.2820747691708223
RMSE Position Ball 2: 0.357495210493538
RMSE Velocity Ball 1: 2.478510768486282
RMSE Velocity Ball 2: 2.613852448919243
